# FINS notebook 6 — fractional Burgers: learned operators (Colab, GPU)

Registered in `README_EXPERIMENTS.md` (exp57, FB5, 08.10) before any of these runs. The equation
u_t + u u_x = −ν Λ^α u on the torus, α = 1.25, 1.5, 1.75 (nonlocal dissipation) and α = 2 (viscous Burgers, last):
* cell 2: for each α and seed 42–44, an FNO stepper trained on 1024 clean 'mixed' trajectories (exp39's recipe,
  ~15–25 min each), checked by pointwise FINS on data seed 1007, then FINS weak and pointwise through it on the
  evaluation data (exp57's own job list, so the commands are the registered ones). Results arrive stepper by stepper.
* cell 3 shows progress; cell 4 zips the outputs and the steppers into `MyDrive/fins_nb6.zip`.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down). If the
GPU budget runs out, run cell 4 with what exists: α = 2 is last on purpose. After a disconnect rerun cell 1 and cell 2:
finished runs are skipped and a stepper training resumes from its partial state.

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp57), dependencies
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(R, exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp57(z):
    with zipfile.ZipFile(z) as zz:
        return 'fins_repo/experiments/exp57_fburgers.py' in zz.namelist()


ZIP_PATH = newest('fins_repo_full*.zip', has_exp57)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp57: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
sys.path.insert(0, EXPDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator', 'torchdiffeq'], check=False)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun this cell')
import exp57_fburgers as e57
RUNS = os.path.join(R, 'fb_runs')
PDES = ('fb125', 'fb150', 'fb175', 'fb200')


def go(out, cmd):
    if os.path.exists(out):
        print('done', os.path.relpath(out, R))
        return
    os.makedirs(os.path.join(os.path.dirname(out), 'logs'), exist_ok=True)
    log = os.path.join(os.path.dirname(out), 'logs', os.path.basename(out).replace('.npz', '.log'))
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        rc = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    keep = [l.strip() for l in open(log).read().splitlines() if 'algebra inside' in l or 'own count' in l]
    print(os.path.relpath(out, R), 'exit', rc, f'{time.time() - t0:.0f} s', *[k[:120] for k in keep[-2:]], sep='\n   ')

In [ ]:
# 2. per alpha and seed: the stepper (trained if missing, checked on data seed 1007), then FINS weak and pointwise
#    through it (exp57's registered commands)
os.makedirs(os.path.join(R, 'steppers'), exist_ok=True)
for pde in PDES:
    for s in (42, 43, 44):
        chk = os.path.join(R, 'steppers', f'pointwise_{pde}_mixed_stepper_s{s}.npz')
        go(chk, ['exp39_localised.py', '--pde', pde, '--op', 'stepper', '--train-ic', 'mixed', '--train-if-missing',
                 '--seed', str(s), '--data-seed', '1007', '--ic', 'periodic', '--n', '10', '--K', '16', '--every', '16',
                 '--interior', '--ut', 'fd', '--save', chk])
        for f, cmd in e57.jobs([pde], ['clean'], [s], ['weak-learned', 'pointwise-learned'], RUNS):
            go(f, cmd)

In [ ]:
# 3. progress
want = [os.path.join(RUNS, 'clean', f'{k}-learned_{p}_s{s}.npz') for p in PDES for s in (42, 43, 44)
        for k in ('weak', 'pointwise')]
want += [os.path.join(R, f'S_theta_{p}_s{s}_mixed_stepper.pt') for p in PDES for s in (42, 43, 44)]
done = [w for w in want if os.path.exists(w)]
print(f'{len(done)} / {len(want)} done')
for w in want:
    if w not in done:
        print('missing', os.path.relpath(w, R))
for f in sorted(glob.glob(os.path.join(R, 'steppers', 'pointwise_fb*.log'))):
    ep = [l for l in open(f).read().splitlines() if 'epoch' in l]
    print('  ', os.path.basename(f), '|', ep[-1].strip()[:100] if ep else '')

In [ ]:
# 4. zip the outputs and the steppers: MyDrive/fins_nb6.zip
out = '/content/drive/MyDrive/fins_nb6.zip'
d = os.path.join(RUNS, 'clean')
files = glob.glob(os.path.join(d, '*-learned_fb*.npz')) + glob.glob(os.path.join(d, 'logs', '*-learned_fb*.log'))
files += glob.glob(os.path.join(R, 'S_theta_fb*_mixed_stepper.pt'))
files += glob.glob(os.path.join(R, 'steppers', 'pointwise_fb*_mixed_stepper_s4*'))
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(set(files)):
        z.write(f, os.path.relpath(f, R))
print(out, os.path.getsize(out), 'bytes;', len(set(files)), 'files')